# Project FORESIGHT

# Notebook 4: Feature Engineering for Forecasting

## Business Context

Accurate demand forecasting requires more than historical sales.

This notebook creates predictive features from sales history, calendar information, promotions, inventory, stores, and customer behavior.

The engineered dataset produced in this notebook will be used for machine learning forecasting models in the next notebook.

---

## Objectives

This notebook will:

- Load cleaned datasets
- Prepare forecasting dataset
- Create calendar features
- Create lag features
- Create rolling statistics
- Create promotion features
- Create inventory features
- Create store features
- Create customer features
- Save final forecasting dataset

In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

plt.style.use("ggplot")

pd.set_option("display.max_columns", None)

In [2]:
base_path = "/kaggle/input/datasets/mrayyanshehzad/synthetic-retail-dataset-10-million-transactions/retail_clean_dataset/"

customers = pd.read_csv(base_path + "customer_master.csv")
inventory = pd.read_csv(base_path + "inventory_snapshot.csv")
promotions = pd.read_csv(base_path + "promotions.csv")
sales = pd.read_csv(base_path + "sales_transactions.csv")
sku = pd.read_csv(base_path + "sku_master.csv")
stores = pd.read_csv(base_path + "store_master.csv")
flags = pd.read_csv(base_path + "sku_inventory_flags.csv")

print("Datasets Loaded Successfully")

Datasets Loaded Successfully


In [3]:
sales["date"] = pd.to_datetime(sales["date"])

customers["registration_date"] = pd.to_datetime(
    customers["registration_date"]
)

inventory["last_restock_date"] = pd.to_datetime(
    inventory["last_restock_date"]
)

promotions["start_date"] = pd.to_datetime(
    promotions["start_date"]
)

promotions["end_date"] = pd.to_datetime(
    promotions["end_date"]
)

stores["opening_date"] = pd.to_datetime(
    stores["opening_date"]
)

flags["window_start"] = pd.to_datetime(
    flags["window_start"]
)

flags["window_end"] = pd.to_datetime(
    flags["window_end"]
)

sales = sales.drop_duplicates()

print("Data Prepared Successfully")

Data Prepared Successfully


# 1. Calendar Feature Engineering

## Business Question

Can seasonal and calendar-based patterns improve sales forecasting?

Calendar features help forecasting models capture recurring trends such as
weekends, monthly seasonality, quarterly effects, and yearly trends. These
features allow machine learning models to better understand how customer
purchasing behavior changes over time.

In [4]:
# ===========================================
# Calendar Features
# ===========================================

forecast_df = sales.copy()

forecast_df["year"] = forecast_df["date"].dt.year
forecast_df["month"] = forecast_df["date"].dt.month
forecast_df["quarter"] = forecast_df["date"].dt.quarter
forecast_df["day"] = forecast_df["date"].dt.day
forecast_df["day_of_week"] = forecast_df["date"].dt.dayofweek
forecast_df["week_of_year"] = forecast_df["date"].dt.isocalendar().week.astype(int)

forecast_df["is_weekend"] = (
    forecast_df["day_of_week"] >= 5
).astype(int)

forecast_df["is_month_start"] = forecast_df["date"].dt.is_month_start.astype(int)
forecast_df["is_month_end"] = forecast_df["date"].dt.is_month_end.astype(int)

forecast_df["is_quarter_start"] = forecast_df["date"].dt.is_quarter_start.astype(int)
forecast_df["is_quarter_end"] = forecast_df["date"].dt.is_quarter_end.astype(int)

forecast_df["is_year_start"] = forecast_df["date"].dt.is_year_start.astype(int)
forecast_df["is_year_end"] = forecast_df["date"].dt.is_year_end.astype(int)

forecast_df.head()

,date,receipt_id,store_id,sku_id,customer_id,quantity,unit_price,total_value,channel,discount_pct,promo_id,year,month,quarter,day,day_of_week,week_of_year,is_weekend,is_month_start,is_month_end,is_quarter_start,is_quarter_end,is_year_start,is_year_end
0,2025-04-02,RCPT00000001,ST16,SKU02498,CUST01410,1,2379.11,2379.11,In-Store,0.0,NaN,2025,4,2,2,2,14,0,0,0,0,0,0,0
1,2025-04-02,RCPT00000001,ST16,SKU04596,CUST01410,3,335.55,1006.65,In-Store,0.0,NaN,2025,4,2,2,2,14,0,0,0,0,0,0,0
2,2022-04-24,RCPT00000002,ST15,SKU00078,CUST00134,1,820.53,820.53,Online,0.0,NaN,2022,4,2,24,6,16,1,0,0,0,0,0,0
3,2022-04-24,RCPT00000002,ST15,SKU00554,CUST00134,2,88.32,176.64,Online,0.0,NaN,2022,4,2,24,6,16,1,0,0,0,0,0,0
4,2024-09-22,RCPT00000003,ST20,SKU03727,CUST08826,1,1660.93,1660.93,Online,0.0,NaN,2024,9,3,22,6,38,1,0,0,0,0,0,0


# 2. Prepare Forecasting Dataset

## Business Question

How can transaction-level retail data be transformed into a dataset suitable for forecasting?

Machine learning forecasting models require data organized over time. Instead of individual transactions, we aggregate sales at the Store–SKU–Day level. This creates one record per product per store per day, allowing the model to learn demand patterns over time.

In [5]:
# ===========================================
# Prepare Forecasting Dataset
# ===========================================

forecast_df = (
    sales
    .groupby(
        ["date", "store_id", "sku_id"],
        as_index=False
    )
    .agg(
        daily_quantity=("quantity", "sum"),
        daily_revenue=("total_value", "sum"),
        avg_price=("unit_price", "mean"),
        avg_discount=("discount_pct", "mean"),
        transactions=("receipt_id", "nunique")
    )
)

forecast_df.head()

,date,store_id,sku_id,daily_quantity,daily_revenue,avg_price,avg_discount,transactions
0,2022-01-01,ST01,SKU00004,1,233.64,233.64,0.0,1
1,2022-01-01,ST01,SKU00059,1,122.45,122.45,0.0,1
2,2022-01-01,ST01,SKU00099,2,3343.98,1671.99,0.0,1
3,2022-01-01,ST01,SKU00130,1,362.18,362.18,0.0,1
4,2022-01-01,ST01,SKU00233,4,483.36,120.84,0.0,1


In [6]:
forecast_df.shape

(8528538, 8)

In [7]:
forecast_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8528538 entries, 0 to 8528537
Data columns (total 8 columns):
 #   Column          Dtype         
---  ------          -----         
 0   date            datetime64[ns]
 1   store_id        object        
 2   sku_id          object        
 3   daily_quantity  int64         
 4   daily_revenue   float64       
 5   avg_price       float64       
 6   avg_discount    float64       
 7   transactions    int64         
dtypes: datetime64[ns](1), float64(3), int64(2), object(2)
memory usage: 520.5+ MB


# 3. Lag Feature Engineering

## Business Question

Can historical sales improve future demand forecasting?

Lag features use previous sales values as predictor variables. They allow forecasting models to recognize short-term demand changes, weekly seasonality, and long-term purchasing behavior.

In [8]:
# ===========================================
# Lag Features
# ===========================================

forecast_df = forecast_df.sort_values(
    ["store_id", "sku_id", "date"]
)

In [9]:
forecast_df["lag_1"] = (
    forecast_df
    .groupby(["store_id", "sku_id"])["daily_quantity"]
    .shift(1)
)

In [10]:
forecast_df["lag_7"] = (
    forecast_df
    .groupby(["store_id", "sku_id"])["daily_quantity"]
    .shift(7)
)

In [11]:
forecast_df["lag_14"] = (
    forecast_df
    .groupby(["store_id", "sku_id"])["daily_quantity"]
    .shift(14)
)

In [12]:
forecast_df["lag_28"] = (
    forecast_df
    .groupby(["store_id", "sku_id"])["daily_quantity"]
    .shift(28)
)

In [13]:
forecast_df[
    [
        "date",
        "store_id",
        "sku_id",
        "daily_quantity",
        "lag_1",
        "lag_7",
        "lag_14",
        "lag_28"
    ]
].head(20)

,date,store_id,sku_id,daily_quantity,lag_1,lag_7,lag_14,lag_28
62577,2022-01-15,ST01,SKU00001,3,NaN,NaN,NaN,NaN
176632,2022-02-09,ST01,SKU00001,4,3.0,NaN,NaN,NaN
412504,2022-03-30,ST01,SKU00001,1,4.0,NaN,NaN,NaN
442752,2022-04-05,ST01,SKU00001,2,1.0,NaN,NaN,NaN
616151,2022-05-09,ST01,SKU00001,3,2.0,NaN,NaN,NaN
631689,2022-05-12,ST01,SKU00001,1,3.0,NaN,NaN,NaN
1298207,2022-09-15,ST01,SKU00001,3,1.0,NaN,NaN,NaN
1404240,2022-10-04,ST01,SKU00001,1,3.0,3.0,NaN,NaN
1661800,2022-11-17,ST01,SKU00001,2,1.0,4.0,NaN,NaN
1802893,2022-12-08,ST01,SKU00001,1,2.0,1.0,NaN,NaN


# 4. Rolling Statistics

## Business Question

Can recent sales trends improve demand forecasting?

Rolling statistics summarize recent sales behavior over a moving time window. They help the forecasting model capture short-term trends, smooth daily fluctuations, and identify changes in customer demand over time.

In [14]:
# ===========================================
# Rolling Statistics
# ===========================================

forecast_df["rolling_mean_7"] = (
    forecast_df
    .groupby(["store_id", "sku_id"])["daily_quantity"]
    .transform(lambda x: x.shift(1).rolling(window=7).mean())
)

forecast_df["rolling_std_7"] = (
    forecast_df
    .groupby(["store_id", "sku_id"])["daily_quantity"]
    .transform(lambda x: x.shift(1).rolling(window=7).std())
)

forecast_df["rolling_mean_30"] = (
    forecast_df
    .groupby(["store_id", "sku_id"])["daily_quantity"]
    .transform(lambda x: x.shift(1).rolling(window=30).mean())
)

forecast_df["rolling_std_30"] = (
    forecast_df
    .groupby(["store_id", "sku_id"])["daily_quantity"]
    .transform(lambda x: x.shift(1).rolling(window=30).std())
)

forecast_df.head()

,date,store_id,sku_id,daily_quantity,daily_revenue,avg_price,avg_discount,transactions,lag_1,lag_7,lag_14,lag_28,rolling_mean_7,rolling_std_7,rolling_mean_30,rolling_std_30
62577,2022-01-15,ST01,SKU00001,3,2440.23,813.41,0.0,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
176632,2022-02-09,ST01,SKU00001,4,3253.64,813.41,0.0,1,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
412504,2022-03-30,ST01,SKU00001,1,813.41,813.41,0.0,1,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
442752,2022-04-05,ST01,SKU00001,2,1626.82,813.41,0.0,1,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
616151,2022-05-09,ST01,SKU00001,3,2440.23,813.41,0.0,1,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [15]:
forecast_df[
    [
        "date",
        "store_id",
        "sku_id",
        "daily_quantity",
        "rolling_mean_7",
        "rolling_std_7",
        "rolling_mean_30",
        "rolling_std_30"
    ]
].head(20)

,date,store_id,sku_id,daily_quantity,rolling_mean_7,rolling_std_7,rolling_mean_30,rolling_std_30
62577,2022-01-15,ST01,SKU00001,3,NaN,NaN,NaN,NaN
176632,2022-02-09,ST01,SKU00001,4,NaN,NaN,NaN,NaN
412504,2022-03-30,ST01,SKU00001,1,NaN,NaN,NaN,NaN
442752,2022-04-05,ST01,SKU00001,2,NaN,NaN,NaN,NaN
616151,2022-05-09,ST01,SKU00001,3,NaN,NaN,NaN,NaN
631689,2022-05-12,ST01,SKU00001,1,NaN,NaN,NaN,NaN
1298207,2022-09-15,ST01,SKU00001,3,NaN,NaN,NaN,NaN
1404240,2022-10-04,ST01,SKU00001,1,2.428571,1.133893,NaN,NaN
1661800,2022-11-17,ST01,SKU00001,2,2.142857,1.214986,NaN,NaN
1802893,2022-12-08,ST01,SKU00001,1,1.857143,0.899735,NaN,NaN


# 5. Inventory Feature Engineering

## Business Question

Can inventory availability improve demand forecasting?

Inventory levels influence product sales and demand patterns. By incorporating stock information into the forecasting dataset, the model can distinguish between low sales caused by reduced demand and low sales caused by stock shortages.

In [16]:
# ===========================================
# Inventory Feature Engineering
# ===========================================

inventory_features = inventory[
    [
        "store_id",
        "sku_id",
        "last_restock_date",
        "stock_on_hand",
        "reorder_point"
    ]
].copy()

forecast_df = forecast_df.merge(
    inventory_features,
    on=["store_id", "sku_id"],
    how="left"
)

forecast_df.head()

,date,store_id,sku_id,daily_quantity,daily_revenue,avg_price,avg_discount,transactions,lag_1,lag_7,lag_14,lag_28,rolling_mean_7,rolling_std_7,rolling_mean_30,rolling_std_30,last_restock_date,stock_on_hand,reorder_point
0,2022-01-15,ST01,SKU00001,3,2440.23,813.41,0.0,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN
1,2022-02-09,ST01,SKU00001,4,3253.64,813.41,0.0,1,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN
2,2022-03-30,ST01,SKU00001,1,813.41,813.41,0.0,1,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN
3,2022-04-05,ST01,SKU00001,2,1626.82,813.41,0.0,1,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN
4,2022-05-09,ST01,SKU00001,3,2440.23,813.41,0.0,1,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN


In [17]:
forecast_df["stock_cover"] = (
    forecast_df["stock_on_hand"] /
    forecast_df["rolling_mean_7"].replace(0, np.nan)
)

In [18]:
forecast_df["low_stock_flag"] = (
    forecast_df["stock_on_hand"] <= forecast_df["reorder_point"]
).astype(int)

In [19]:
forecast_df[
    [
        "store_id",
        "sku_id",
        "stock_on_hand",
        "reorder_point",
        "stock_cover",
        "low_stock_flag"
    ]
].head(20)

,store_id,sku_id,stock_on_hand,reorder_point,stock_cover,low_stock_flag
0,ST01,SKU00001,NaN,NaN,NaN,0
1,ST01,SKU00001,NaN,NaN,NaN,0
2,ST01,SKU00001,NaN,NaN,NaN,0
3,ST01,SKU00001,NaN,NaN,NaN,0
4,ST01,SKU00001,NaN,NaN,NaN,0
5,ST01,SKU00001,NaN,NaN,NaN,0
6,ST01,SKU00001,NaN,NaN,NaN,0
7,ST01,SKU00001,NaN,NaN,NaN,0
8,ST01,SKU00001,NaN,NaN,NaN,0
9,ST01,SKU00001,NaN,NaN,NaN,0


In [20]:
inventory.columns.tolist()

['store_id',
 'sku_id',
 'stock_on_hand',
 'reorder_point',
 'safety_stock',
 'last_restock_date']

In [21]:
inventory.head()

,store_id,sku_id,stock_on_hand,reorder_point,safety_stock,last_restock_date
0,ST11,SKU02558,323,72,22,2025-05-17
1,ST21,SKU01031,236,67,14,2025-06-17
2,ST26,SKU02129,433,96,34,2025-05-20
3,ST19,SKU02907,178,34,13,2025-08-03
4,ST05,SKU01023,333,97,22,2025-07-12


In [22]:
print("Forecast Store IDs:")
print(forecast_df["store_id"].unique()[:5])

print("\nInventory Store IDs:")
print(inventory["store_id"].unique()[:5])

print("\nForecast SKU IDs:")
print(forecast_df["sku_id"].unique()[:5])

print("\nInventory SKU IDs:")
print(inventory["sku_id"].unique()[:5])

Forecast Store IDs:
['ST01' 'ST02' 'ST03' 'ST04' 'ST05']

Inventory Store IDs:
['ST11' 'ST21' 'ST26' 'ST19' 'ST05']

Forecast SKU IDs:
['SKU00001' 'SKU00002' 'SKU00003' 'SKU00004' 'SKU00005']

Inventory SKU IDs:
['SKU02558' 'SKU01031' 'SKU02129' 'SKU02907' 'SKU01023']


In [23]:
print(forecast_df[["store_id", "sku_id"]].dtypes)
print(inventory[["store_id", "sku_id"]].dtypes)

store_id    object
sku_id      object
dtype: object
store_id    object
sku_id      object
dtype: object


# 6. Price Feature Engineering

## Business Question

Can pricing and discounts influence customer demand?

Product pricing and promotional discounts play a significant role in purchasing behavior. By creating price-related features, forecasting models can better understand how changes in pricing impact future demand.

In [24]:
# ===========================================
# Price Feature Engineering
# ===========================================

forecast_df["price_per_unit"] = forecast_df["avg_price"]

forecast_df["discount_rate"] = forecast_df["avg_discount"]

forecast_df.head()

,date,store_id,sku_id,daily_quantity,daily_revenue,avg_price,avg_discount,transactions,lag_1,lag_7,lag_14,lag_28,rolling_mean_7,rolling_std_7,rolling_mean_30,rolling_std_30,last_restock_date,stock_on_hand,reorder_point,stock_cover,low_stock_flag,price_per_unit,discount_rate
0,2022-01-15,ST01,SKU00001,3,2440.23,813.41,0.0,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,0,813.41,0.0
1,2022-02-09,ST01,SKU00001,4,3253.64,813.41,0.0,1,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,0,813.41,0.0
2,2022-03-30,ST01,SKU00001,1,813.41,813.41,0.0,1,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,0,813.41,0.0
3,2022-04-05,ST01,SKU00001,2,1626.82,813.41,0.0,1,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,0,813.41,0.0
4,2022-05-09,ST01,SKU00001,3,2440.23,813.41,0.0,1,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,0,813.41,0.0


In [25]:
forecast_df["price_change"] = (
    forecast_df
    .groupby(["store_id", "sku_id"])["price_per_unit"]
    .diff()
)

In [26]:
forecast_df["price_change_pct"] = (
    forecast_df
    .groupby(["store_id", "sku_id"])["price_per_unit"]
    .pct_change()
)

In [27]:
forecast_df["promotion_flag"] = (
    forecast_df["discount_rate"] > 0
).astype(int)

In [28]:
forecast_df["discount_category"] = pd.cut(
    forecast_df["discount_rate"],
    bins=[-0.01, 0, 0.10, 0.20, 1],
    labels=[
        "No Discount",
        "Low",
        "Medium",
        "High"
    ]
)

In [29]:
forecast_df[
    [
        "store_id",
        "sku_id",
        "price_per_unit",
        "discount_rate",
        "price_change",
        "price_change_pct",
        "promotion_flag",
        "discount_category"
    ]
].head(20)

,store_id,sku_id,price_per_unit,discount_rate,price_change,price_change_pct,promotion_flag,discount_category
0,ST01,SKU00001,813.41,0.0,NaN,NaN,0,No Discount
1,ST01,SKU00001,813.41,0.0,0.0,0.0,0,No Discount
2,ST01,SKU00001,813.41,0.0,0.0,0.0,0,No Discount
3,ST01,SKU00001,813.41,0.0,0.0,0.0,0,No Discount
4,ST01,SKU00001,813.41,0.0,0.0,0.0,0,No Discount
5,ST01,SKU00001,813.41,0.0,0.0,0.0,0,No Discount
6,ST01,SKU00001,813.41,0.0,0.0,0.0,0,No Discount
7,ST01,SKU00001,813.41,0.0,0.0,0.0,0,No Discount
8,ST01,SKU00001,813.41,0.0,0.0,0.0,0,No Discount
9,ST01,SKU00001,813.41,0.0,0.0,0.0,0,No Discount


# 7. Final Dataset Preparation

## Business Question

How can we prepare the engineered dataset for machine learning?

Before training forecasting models, missing values generated during lag and rolling calculations should be handled appropriately. This ensures that the final dataset is clean and ready for model development.

In [30]:
forecast_df.isnull().sum().sort_values(ascending=False)

stock_cover          7030098
stock_on_hand        6881598
last_restock_date    6881598
reorder_point        6881598
rolling_std_30       4004233
rolling_mean_30      4004233
lag_28               3812847
lag_14               2085279
discount_category    1795287
lag_7                1049773
rolling_mean_7       1049773
rolling_std_7        1049773
lag_1                 150000
price_change          150000
price_change_pct      150000
daily_quantity             0
date                       0
store_id                   0
sku_id                     0
transactions               0
daily_revenue              0
avg_price                  0
avg_discount               0
discount_rate              0
price_per_unit             0
low_stock_flag             0
promotion_flag             0
dtype: int64

In [31]:
forecast_df = forecast_df.dropna().reset_index(drop=True)

In [32]:
print("Final Shape:", forecast_df.shape)

forecast_df.isnull().sum()

Final Shape: (873633, 27)


date                 0
store_id             0
sku_id               0
daily_quantity       0
daily_revenue        0
avg_price            0
avg_discount         0
transactions         0
lag_1                0
lag_7                0
lag_14               0
lag_28               0
rolling_mean_7       0
rolling_std_7        0
rolling_mean_30      0
rolling_std_30       0
last_restock_date    0
stock_on_hand        0
reorder_point        0
stock_cover          0
low_stock_flag       0
price_per_unit       0
discount_rate        0
price_change         0
price_change_pct     0
promotion_flag       0
discount_category    0
dtype: int64

In [33]:
forecast_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 873633 entries, 0 to 873632
Data columns (total 27 columns):
 #   Column             Non-Null Count   Dtype         
---  ------             --------------   -----         
 0   date               873633 non-null  datetime64[ns]
 1   store_id           873633 non-null  object        
 2   sku_id             873633 non-null  object        
 3   daily_quantity     873633 non-null  int64         
 4   daily_revenue      873633 non-null  float64       
 5   avg_price          873633 non-null  float64       
 6   avg_discount       873633 non-null  float64       
 7   transactions       873633 non-null  int64         
 8   lag_1              873633 non-null  float64       
 9   lag_7              873633 non-null  float64       
 10  lag_14             873633 non-null  float64       
 11  lag_28             873633 non-null  float64       
 12  rolling_mean_7     873633 non-null  float64       
 13  rolling_std_7      873633 non-null  float64 

In [34]:
forecast_df.head()

,date,store_id,sku_id,daily_quantity,daily_revenue,avg_price,avg_discount,transactions,lag_1,lag_7,lag_14,lag_28,rolling_mean_7,rolling_std_7,rolling_mean_30,rolling_std_30,last_restock_date,stock_on_hand,reorder_point,stock_cover,low_stock_flag,price_per_unit,discount_rate,price_change,price_change_pct,promotion_flag,discount_category
0,2025-10-25,ST01,SKU00005,4,554.0,138.5,0.0,1,3.0,2.0,1.0,3.0,2.142857,1.069045,1.800000,1.186127,2025-11-07,15.0,91.0,7.000000,1,138.5,0.0,0.0,0.0,0,No Discount
1,2025-11-01,ST01,SKU00005,5,692.5,138.5,0.0,1,4.0,1.0,4.0,1.0,2.428571,1.272418,1.866667,1.252125,2025-11-07,15.0,91.0,6.176471,1,138.5,0.0,0.0,0.0,0,No Discount
2,2025-11-07,ST01,SKU00005,3,415.5,138.5,0.0,1,5.0,4.0,1.0,3.0,3.000000,1.414214,1.900000,1.322224,2025-11-07,15.0,91.0,5.000000,1,138.5,0.0,0.0,0.0,0,No Discount
3,2022-09-23,ST01,SKU00020,3,328.5,109.5,0.0,1,3.0,1.0,2.0,6.0,2.000000,0.816497,2.133333,1.252125,2025-10-06,0.0,37.0,0.000000,1,109.5,0.0,0.0,0.0,0,No Discount
4,2022-09-27,ST01,SKU00020,5,547.5,109.5,0.0,1,3.0,2.0,1.0,4.0,2.285714,0.755929,2.166667,1.261727,2025-10-06,0.0,37.0,0.000000,1,109.5,0.0,0.0,0.0,0,No Discount


## Feature Summary

The final forecasting dataset includes:

- Calendar Features
- Lag Features
- Rolling Statistics
- Inventory Features
- Price Features
- Promotion Features

This engineered dataset will be used in Notebook 5 for machine learning model training and demand forecasting.

# 8. Save Feature Engineered Dataset

The final forecasting dataset is saved for use in Notebook 5, where machine learning models will be trained and evaluated.

In [35]:
forecast_df.to_csv(
    "forecast_feature_engineered.csv",
    index=False
)

print("Forecast dataset saved successfully!")

Forecast dataset saved successfully!


In [36]:
import os

print(os.getcwd())
print(os.listdir("/kaggle/working"))

/kaggle/working
['__notebook__.ipynb', 'forecast_feature_engineered.csv']
